# 00 · Dựng GNN-RAG features (ReaRev) cho WebQSP và CWQ

Bước chuẩn bị dữ liệu cho phần KG-RAG. Mỗi câu hỏi được lưu thành một bản ghi gồm: câu hỏi, thực thể chủ đề, subgraph Freebase,
điểm ReaRev (GNN) cho từng thực thể ứng viên và đáp án gold. Mọi phương pháp ở notebook 01 đọc chung các file này.

| Output | Nội dung |
|---|---|
| `ckpt/gnn_features_train.pkl` | WebQSP train, 2848 câu |
| `ckpt/gnn_features_webqsp_test.pkl` | WebQSP test, 1639 câu |
| `ckpt/gnn_features_cwq_test.pkl` | CWQ test, 3531 câu (tuỳ chọn) |

Nếu các file đã có trên Drive thì chỉ cần chạy 00a, 00b, 01a, 01d để kiểm tra. 01b và 01c chỉ dùng khi phải dựng lại từ repo GNN-RAG.

## 00 · Thiết lập

In [ ]:
# ==== 00a. Mount Drive + đường dẫn (SỬA NẾU KHÁC) ====
from google.colab import drive; drive.mount('/content/drive')
DRIVE_BASE  = '/content/drive/MyDrive/ThS/Research/colab'          # = thư mục colab/ local
CODE_DIR    = f'{DRIVE_BASE}/seminar2_kgrag'                          # folder code này
GRAPH_RAG   = f'{DRIVE_BASE}/graph_rag'                               # dữ liệu, checkpoint, kết quả
RESULTS     = f'{GRAPH_RAG}/results'
CKPT        = f'{GRAPH_RAG}/ckpt'
GNN_RAG_DIR = '/content/drive/MyDrive/ThS/Research/GNN-RAG'          # repo GNN-RAG (chỉ cần nếu phải dựng lại pkl)
RUN_DIR     = f'{DRIVE_BASE}/run'                                      # chứa checkpoint.zip (ReaRev_webqsp/CWQ.ckpt) và data zip tải từ README GNN-RAG
# Thư mục checkpoint CŨ của Seminar 1 (sẽ copy pkl từ đây nếu ckpt/ chưa có):
OLD_CKPT_DIRS = ['/content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING/checkpoints',
                 '/content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING/utils/checkpoints']

import os, sys, json, subprocess, time, shutil, glob
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; sys.dont_write_bytecode = True   # 2 runtime cùng import từ Drive → không ghi .pyc
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
import pickle
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
print('cwd =', os.getcwd())
assert os.path.exists('kgqa_bench/cli.py'), 'chưa upload folder seminar2_kgrag lên Drive'

In [ ]:
# ==== 00b. Deps + GPU + import check ====
!pip install -q datasets sentence-transformers transformers accelerate scikit-learn networkx tqdm matplotlib tabulate
import torch, pandas as pd
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
from kgqa_bench.data import load_spec, load_rog_hf, BenchDataset
from kgqa_bench.metrics import summarize_records, compare_records, matched_cost_table
print('kgqa_bench OK')

def run_bench(out, train=None, evals=(), methods=(), extra='', force=False):
    if os.path.exists(f'{out}/results.json') and not force:
        print(f'[skip] {out} đã có results.json'); return load_results(out)
    cmd = f'python -m kgqa_bench.cli run --device {DEVICE} --out {out} '
    if train: cmd += f'--train {train} '
    cmd += '--eval ' + ' '.join(evals) + ' --methods ' + ' '.join(methods) + ' ' + extra
    print('$', cmd); t0 = time.time()
    rc = subprocess.call(cmd, shell=True)
    print(f'exit={rc}  {(time.time()-t0)/60:.1f} min')
    return load_results(out)

def load_results(out):
    with open(f'{out}/results.json') as f: return json.load(f)

COLS = ['n','f1','cov','prec','hit_ctx','cost','n_tokens','latency_ms','ece','aurc','sel_acc@80','las_hit1']
def show(R, cols=COLS):
    for ds, by_m in R['summaries'].items():
        df = pd.DataFrame(by_m).T; df = df[[c for c in cols if c in df.columns]]
        print(f'\n=== {ds} ==='); display(df.round(3))

def show_cmp(R):
    rows = []
    for ds, pairs in R['comparisons'].items():
        for k, c in pairs.items():
            if not c or not c.get('n'): continue
            b = c['bootstrap']
            rows.append(dict(dataset=ds, pair=k, n=c['n'], delta=round(b['delta']*100,2),
                             ci=f"[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}]", p=round(b['p_value'],4),
                             mcnemar_p=round(c.get('mcnemar',{}).get('p_value', float('nan')),4), sig=c['significant_0.05']))
    display(pd.DataFrame(rows))

def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
def records(out, ds_key, method):
    p = f'{out}/records_{ds_key.replace(":", "_")}_{method}.jsonl'
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

# Mốc Seminar 1 (WebQSP test, GNN features thật, A100) — chỉ để đối chiếu
SEM1 = {'gnnrag_all': dict(f1=.400, cov=.694, prec=.339, cost=6.5),
        'gnnrag_top10': dict(f1=.396, cost=5.1),
        'cgar': dict(f1=.407, cov=.688, prec=.353, cost=5.1),
        'ceiling': .73}

## 01 · Tìm hoặc dựng GNN-RAG features cho WebQSP
**Mục tiêu:** có 2 file `ckpt/gnn_features_train.pkl` (WebQSP train, từ `train.info` do P2 cell 18 tạo) và `ckpt/gnn_features_webqsp_test.pkl` (từ `test.info` GNN-RAG phát hành). Thứ tự tìm: `ckpt/` → thư mục checkpoint Seminar 1 (copy sang) → dựng lại từ repo GNN-RAG.
**Output kỳ vọng:** train ≈ 2.8k câu (qid `WebQTrn-…`), test 1628 câu (qid `WebQTest-…`); `answer_in_subgraph_rate` test ≈ 0.70–0.80 (Seminar 1: 0.73); `mean_n_candidates` = 20; `gnn_hit` trung bình ≈ 0.85–0.90 (ReaRev Hit@1 trong paper ≈ 0.80 Hit).
**Gate:** qid train ∩ test = ∅; trần coverage trong [0.65, 0.85]. Trần < 0.5 → pkl sai (tên thực thể chưa resolve) → dựng lại.
**Lập luận:** trần coverage của pool GNN (≈0.73) so với pool heuristic (hit_ctx 0.33) là lý do duy nhất số tuyệt đối sẽ tăng; CGAR không đổi.

In [ ]:
# ==== 01a. Tìm pkl có sẵn; thiếu thì copy từ Seminar 1 ====
WANT = {'train': ['gnn_features_train.pkl'],
        'test':  ['gnn_features_webqsp_test.pkl', 'gnn_features_test.pkl']}
FEAT = {}
for split, names in WANT.items():
    for n in names:
        p = f'{CKPT}/{n}'
        if os.path.exists(p): FEAT[split] = p; break
    if split not in FEAT:
        for d in OLD_CKPT_DIRS:
            for n in names:
                src = f'{d}/{n}'
                if os.path.exists(src):
                    dst = f'{CKPT}/{names[0]}'; print(f'copy {src} -> {dst}'); shutil.copy(src, dst); FEAT[split] = dst; break
            if split in FEAT: break
# C2 (11/2026): CWQ với features ReaRev_CWQ. BUILD_CWQ=False để chỉ làm WebQSP như trước.
BUILD_CWQ = True
FEAT_CWQ = {}
for split, n in (('train', 'gnn_features_cwq_train.pkl'), ('test', 'gnn_features_cwq_test.pkl')):
    if os.path.exists(f'{CKPT}/{n}'): FEAT_CWQ[split] = f'{CKPT}/{n}'
print('FEAT =', FEAT, '| FEAT_CWQ =', FEAT_CWQ)
if len(FEAT) < 2:
    print('THIẾU:', [s for s in WANT if s not in FEAT], '→ chạy cell 01b để dựng lại từ repo GNN-RAG')
    print('Các file trong ckpt/:'); print([f for f in os.listdir(CKPT) if f.endswith('.pkl')])

In [ ]:
# ==== 01b. (Chỉ khi 01a thiếu) Dựng pkl từ repo GNN-RAG — port P2 cell 30 / P7v3 cell 2 ====
# Cần: gnn/data/webqsp/{train,test}.json, entities.txt, relations.txt; llm/entities_names.json;
#      llm/results/gnn/RoG-webqsp/rearev-sbert/test.info  (test, GNN-RAG phát hành)
#      gnn/checkpoint/pretrain/train.info                   (train, P2 cell 18 tạo bằng cách eval ReaRev trên train.json)
def build_gnn_pkl(dataset, split, out_path):
    dd = 'webqsp' if dataset == 'webqsp' else 'CWQ'
    data_path = f'{GNN_RAG_DIR}/gnn/data/{dd}/{split}.json'
    info_path = (f'{GNN_RAG_DIR}/llm/results/gnn/RoG-{dataset}/rearev-sbert/test.info' if split == 'test'
                 else f'{GNN_RAG_DIR}/gnn/checkpoint/pretrain/' + ('train.info' if dataset == 'webqsp' else f'train_{dataset}.info'))
    ents, rels, names = f'{GNN_RAG_DIR}/gnn/data/{dd}/entities.txt', f'{GNN_RAG_DIR}/gnn/data/{dd}/relations.txt', f'{GNN_RAG_DIR}/llm/entities_names.json'
    for p in (data_path, info_path, ents, rels, names):
        if not os.path.exists(p): print('MISSING', p); return None
    id2mid = {i: l.strip() for i, l in enumerate(open(ents))}
    id2rel = {i: l.strip() for i, l in enumerate(open(rels))}
    mid2name = json.load(open(names))
    def ent(e):
        if isinstance(e, int): m = id2mid.get(e, str(e)); return mid2name.get(m, m)
        if isinstance(e, str) and e.startswith('m.'): return mid2name.get(e, e)
        return str(e)
    def rel(r): return id2rel.get(r, str(r)) if isinstance(r, int) else str(r)
    data = [json.loads(l) for l in open(data_path)]
    info = [json.loads(l) for l in open(info_path)]
    assert len(data) == len(info), f'{len(data)} data vs {len(info)} info — info không khớp split'
    feats = {}
    for d, g in zip(data, info):
        cand = g.get('cand', [])
        node_scores = {mid2name.get(m, m): s for m, s in cand}
        ans = [a.get('text', mid2name.get(a.get('kb_id', ''), str(a))) if isinstance(a, dict) else ent(a) for a in d.get('answers', [])]
        feats[d['id']] = dict(question=d['question'], node_scores=node_scores,
                              top_candidates=[mid2name.get(c[0], c[0]) for c in cand[:20]],
                              q_entities=[ent(e) for e in d.get('entities', [])], answers=ans,
                              subgraph_tuples=[[ent(h), rel(r), ent(t)] for h, r, t in d['subgraph']['tuples']],
                              gnn_f1=g.get('f1', 0), gnn_hit=g.get('hit', 0))
    pickle.dump(feats, open(out_path, 'wb')); print(f'saved {len(feats)} -> {out_path}'); return out_path

if 'test' not in FEAT:  FEAT['test']  = build_gnn_pkl('webqsp', 'test',  f'{CKPT}/gnn_features_webqsp_test.pkl')
if 'train' not in FEAT:
    if not os.path.exists(f'{GNN_RAG_DIR}/gnn/checkpoint/pretrain/train.info'):
        print('Chưa có train.info → chạy cell 01c (eval ReaRev trên train split, ~10 phút GPU) rồi chạy lại cell này')
    else:
        FEAT['train'] = build_gnn_pkl('webqsp', 'train', f'{CKPT}/gnn_features_train.pkl')
# C2: CWQ — test.info do GNN-RAG phát hành; train_cwq.info do cell 01c tạo bằng ReaRev_CWQ.ckpt
if BUILD_CWQ:
    if 'test' not in FEAT_CWQ:
        r = build_gnn_pkl('cwq', 'test', f'{CKPT}/gnn_features_cwq_test.pkl')
        if r: FEAT_CWQ['test'] = r
    if 'train' not in FEAT_CWQ:
        if os.path.exists(f'{GNN_RAG_DIR}/gnn/checkpoint/pretrain/train_cwq.info'):
            r = build_gnn_pkl('cwq', 'train', f'{CKPT}/gnn_features_cwq_train.pkl')
            if r: FEAT_CWQ['train'] = r
        else:
            print('CWQ: chưa có train_cwq.info → cell 01c sẽ eval ReaRev_CWQ trên CWQ train (~30 phút GPU, 27k câu)')
print('FEAT =', FEAT, '| FEAT_CWQ =', FEAT_CWQ)

In [ ]:
# ==== 01c. (Chỉ khi thiếu train.info) Eval ReaRev trên WebQSP train → train.info — port P2 cell 18 ====
# Cần checkpoint gnn/checkpoint/pretrain/ReaRev_webqsp.ckpt và repo GNN-RAG cài được (torch_geometric...).
import zipfile
g = f'{GNN_RAG_DIR}/gnn'
# Tham số ReaRev theo README GNN-RAG (nếu load ckpt báo lệch shape → chỉnh num_iter/num_ins/num_gnn cho khớp ckpt)
REAREV_ARGS = {'webqsp': dict(folder='webqsp', ckpt='ReaRev_webqsp.ckpt', args='--num_iter 3 --num_ins 2 --num_gnn 3 --num_epoch 200'),
               'cwq':    dict(folder='CWQ',    ckpt='ReaRev_CWQ.ckpt',    args='--num_iter 2 --num_ins 3 --num_gnn 3 --num_epoch 100')}
def ensure_gnnrag_repo():
    if not os.path.exists(f'{g}/main.py'):                     # chưa có repo → clone
        subprocess.call(f'git clone --depth 1 https://github.com/cmavro/GNN-RAG {GNN_RAG_DIR}', shell=True)
    if os.path.exists(f'{RUN_DIR}/checkpoint.zip') and not all(os.path.exists(f'{g}/checkpoint/pretrain/{v["ckpt"]}') for v in REAREV_ARGS.values()):
        with zipfile.ZipFile(f'{RUN_DIR}/checkpoint.zip') as z:   # README GNN-RAG "download released GNN models"
            for m in z.namelist():
                if m.startswith('checkpoint/') and not m.endswith('/') and '.DS_Store' not in m: z.extract(m, g)
        print('ckpt:', os.listdir(f'{g}/checkpoint/pretrain'))
    for folder in ('webqsp', 'CWQ'):
        if not os.path.exists(f'{g}/data/{folder}/train.json'):
            for zf in glob.glob(f'{RUN_DIR}/*.zip'):               # README GNN-RAG "download data" (data.zip có thể lồng trong zip khác)
                if 'checkpoint' in zf: continue
                with zipfile.ZipFile(zf) as z:
                    names = z.namelist()
                    inner = [n for n in names if n.endswith('data.zip')]
                    if inner:
                        z.extract(inner[0], RUN_DIR); zf = f'{RUN_DIR}/{inner[0]}'
                        with zipfile.ZipFile(zf) as z2: names = z2.namelist(); print('giải nén', zf); z2.extractall(g if names[0].startswith('data/') else f'{g}/data')
                    elif any(folder.lower() in n.lower() and n.endswith('train.json') for n in names):
                        print('giải nén data từ', zf); z.extractall(g if names[0].startswith('data/') else f'{g}/data')

def eval_rearev_on_train(dataset):
    """Chạy ReaRev (ckpt công bố) ở chế độ eval trên TRAIN split của `dataset` → gnn/checkpoint/pretrain/train[_cwq].info"""
    a = REAREV_ARGS[dataset]; d = f'{g}/data/{a["folder"]}'
    for p in (f'{g}/checkpoint/pretrain/{a["ckpt"]}', f'{d}/train.json', f'{d}/entities.txt'):
        assert os.path.exists(p), f'thiếu {p} — tải checkpoint.zip / data zip từ README GNN-RAG vào {RUN_DIR}'
    shutil.copy(f'{d}/test.json', f'{d}/test_backup.json'); shutil.copy(f'{d}/train.json', f'{d}/test.json')
    try:
        rc = subprocess.call(f'python main.py ReaRev --entity_dim 50 --batch_size 8 --eval_every 2 --data_folder data/{a["folder"]}/ '
                             f'--lm sbert {a["args"]} --relation_word_emb True --load_experiment {a["ckpt"]} --is_eval --name {dataset}_train_eval',
                             shell=True, cwd=g); print('exit', rc)
    finally:
        shutil.move(f'{d}/test_backup.json', f'{d}/test.json')
    infos = glob.glob(f'{g}/checkpoint/pretrain/*{dataset}_train_eval*.info') or glob.glob(f'{g}/checkpoint/pretrain/*test.info')
    if not infos: print('không thấy .info sau khi eval'); return None
    latest = max(infos, key=os.path.getmtime)
    dst = f'{g}/checkpoint/pretrain/' + ('train.info' if dataset == 'webqsp' else f'train_{dataset}.info')
    shutil.copy(latest, dst); print('->', latest, '=>', dst); return dst

if 'train' not in FEAT and not os.path.exists(f'{g}/checkpoint/pretrain/train.info'):
    ensure_gnnrag_repo()
    if eval_rearev_on_train('webqsp'): FEAT['train'] = build_gnn_pkl('webqsp', 'train', f'{CKPT}/gnn_features_train.pkl')
else:
    print('[skip] WebQSP: đã có train features hoặc train.info')
if BUILD_CWQ and 'train' not in FEAT_CWQ and not os.path.exists(f'{g}/checkpoint/pretrain/train_cwq.info'):
    ensure_gnnrag_repo()
    if eval_rearev_on_train('cwq'):
        r = build_gnn_pkl('cwq', 'train', f'{CKPT}/gnn_features_cwq_train.pkl')
        if r: FEAT_CWQ['train'] = r
    if 'test' not in FEAT_CWQ:
        r = build_gnn_pkl('cwq', 'test', f'{CKPT}/gnn_features_cwq_test.pkl')
        if r: FEAT_CWQ['test'] = r
elif BUILD_CWQ:
    print('[skip] CWQ: đã có features hoặc train_cwq.info')

In [ ]:
# ==== 01d. Kiểm tra pkl qua harness + trần coverage ====
assert 'train' in FEAT and 'test' in FEAT, 'thiếu features — xem 01a/01b/01c'
WEBQSP_TR = f"gnnrag:{FEAT['train']}:webqsp:train"
WEBQSP_TE = f"gnnrag:{FEAT['test']}:webqsp:test"
CWQ_TE    = f"gnnrag:{FEAT_CWQ['test']}:cwq:test" if FEAT_CWQ.get('test') else 'rog:rmanluo/RoG-cwq:test'   # C2: GNN nếu đã dựng
DATA_CACHE = f'{RESULTS}/data_cache'

tr, te = load_spec(WEBQSP_TR), load_spec(WEBQSP_TE)
cwq = load_spec(CWQ_TE) if CWQ_TE.startswith('gnnrag:') else load_rog_hf('rmanluo/RoG-cwq', 'test', cache_dir=DATA_CACHE)
CWQ_TAG = '(gnn)' if CWQ_TE.startswith('gnnrag:') else '(heur)'
stats = {d.key + ('(gnn)' if d is not cwq else CWQ_TAG): d.stats() for d in (tr, te, cwq)}
df = pd.DataFrame(stats).T; display(df.round(3))
import numpy as np
gnn_hit = np.mean([q.meta.get('gnn_hit', 0) for q in te]); print('mean gnn_hit (ReaRev Hit, test) =', round(gnn_hit, 3))
print('qid ví dụ:', tr.qids()[:2], te.qids()[:2])
gate(not (set(tr.qids()) & set(te.qids())), 'train ∩ test = ∅')
gate(0.70 <= gnn_hit <= 0.80, f'mean gnn_hit = {gnn_hit:.3f} ≈ ReaRev Hit 76.4 trong paper GNN-RAG → pkl đúng')
ceil = stats['webqsp:test(gnn)']['answer_in_subgraph_rate']
gate(0.80 <= ceil <= 0.95, f'trần SUBGRAPH WebQSP(gnn) = {ceil:.3f} trong [0.80, 0.95] (subgraph GNN-RAG = subgraph RoG; đo 11/2026: 0.887)')
# Con số thực sự giới hạn recall: đáp án có nằm trong POOL ỨNG VIÊN ReaRev không (Seminar 1 gọi là ~73%)
pool_hit = {k: float(np.mean([any(a in set(q.top_candidates) for a in q.answers) for q in d if q.answers])) for k, d in (('train', tr), ('test', te), ('cwq' + CWQ_TAG, cwq))}
print('answer-in-candidate-pool:', {k: round(v, 3) for k, v in pool_hit.items()})
gate(0.65 <= pool_hit['test'] <= 0.85, f"đáp án trong pool ứng viên GNN (test) = {pool_hit['test']:.3f} (Seminar 1: ≈0.73); heuristic chỉ ≈0.33")
gate(stats['webqsp:test(gnn)']['mean_n_candidates'] >= 3, f"ReaRev xuất trung bình {stats['webqsp:test(gnn)']['mean_n_candidates']:.1f} ứng viên/câu (chỉ ứng viên vượt ngưỡng; 11/2026: 5.6) → pool 10 ≈ pool 20, K của CGAR gộp một mức")
if CWQ_TE.startswith('gnnrag:'):
    cwq_hit = np.mean([q.meta.get('gnn_hit', 0) for q in cwq]); print('mean gnn_hit (ReaRev Hit, CWQ test) =', round(cwq_hit, 3))
    gate(0.45 <= cwq_hit <= 0.62, f'CWQ: mean gnn_hit = {cwq_hit:.3f} ≈ ReaRev Hit 52.9 trong paper GNN-RAG → pkl CWQ đúng')
    gate(pool_hit['cwq(gnn)'] > 0.45, f"CWQ: đáp án trong pool ReaRev = {pool_hit['cwq(gnn)']:.3f} (heuristic ≈0.33)")
stats['pool_hit'] = pool_hit
json.dump(stats, open(f'{RESULTS}/g01_dataset_stats.json', 'w'), indent=2)
print('WEBQSP_TR =', WEBQSP_TR); print('WEBQSP_TE =', WEBQSP_TE); print('CWQ_TE    =', CWQ_TE)